In [27]:
from dotenv import load_dotenv
load_dotenv()

True

In [28]:

from openai import OpenAI
import os

# openai_client = OpenAI(
#     api_key=os.getenv('GROQ_API_KEY'),
#     base_url='https://api.groq.com/openai/v1'
# )

openai_client = OpenAI(
    api_key=os.getenv('GEMINI_API_KEY'),
    base_url='https://generativelanguage.googleapis.com/v1beta/openai/'
)

# Test the client using a Gemini model (e.g., gemini-1.5-flash)
response = openai_client.chat.completions.create(
    model="gemini-3.5-flash",
    messages=[
        {"role": "user", "content": "Hello! Confrim if this works."}
    ]
)

print(response.choices[0].message.content)

Hello! Yes, it works perfectly. I am online and ready to help. How can I assist you today?


In [29]:
# from google import genai
# import os

# # Initialize the client (it automatically picks up the GEMINI_API_KEY env variable)
# googleai_client = genai.Client(
#     api_key=os.getenv('GEMINI_API_KEY'),
#     )

# # Generate a response
# response = googleai_client.models.generate_content(
#     model="gemini-3.5-flash",
#     contents="Explain how AI works in a few words"
# )

# print(response.text)

In [30]:
import requests

docs_url = 'https://datatalks.club/faq/json/courses.json'
response = requests.get(docs_url)
courses_raw = response.json()

In [31]:
documents = []
url_prefix = 'https://datatalks.club/faq'

for course in courses_raw:
    course_url = f'{url_prefix}{course['path']}'

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)
print(documents[100])

{'id': 'c6c716b9e7', 'course': 'machine-learning-zoomcamp', 'section': 'Module 2. Machine Learning for Regression', 'question': 'How to copy a dataframe without changing the original dataframe?', 'answer': 'Copy a dataframe using:\n\n```python\nX_copy = X.copy()\n```\n\nThis creates a deep copy of the dataframe. If you use `X_copy = X`, it will create a "view" and any changes to `X_copy` will affect the original dataframe `X`. This is not a real copy.'}


In [32]:
from minsearch import Index

index = Index(
    text_fields=['question', 'section', 'answer'],
    keyword_fields=['course']
)

index.fit(documents)

In [33]:
question = 'I just discovered the course. Can I join now?'

def search(question, course='llm-zoomcamp'):
    boost_dict = {'question': 2.0, 'section': 0.5}
    filter_dict = {'course': course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

search_results = search(question)

In [34]:
[doc['question'] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'When will the course be offered next?',
 'I missed the first homework - can I still get a certificate?']

In [35]:
INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

In [36]:
USER_PROMPT_TEMPLATE = '''
Question:
{question}

Context:
{context}
'''


In [37]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc['section'])
        lines.append('Q: ' + doc['question'])
        lines.append('A: ' + doc['answer'])
        lines.append('')

    return '\n'.join(lines).strip()

In [38]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [39]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

In [61]:
def llm(instructions, user_prompt, model='gemini-3.5-flash'):
    message_history = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': user_prompt}
    ]

    response = openai_client.chat.completions.create(
        model=model,
        messages=message_history
    )
    return response.choices[0].message.content


In [62]:
def rag(query, model='gemini-3.5-flash'):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [63]:
answer = rag('I just discovered the course. Can I join now?')
print(answer)

Yes, you can still join. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.
